MODEL_NAME = "dinov3_vit_b16"

METHOD_NAME = "dinov3_vit_b16_baseline_expand_10ep"

In [15]:
!rm -rf /content/drive
print("Removed fake local /content/drive")

Removed fake local /content/drive


In [16]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [17]:
import os

print("drive root:", os.listdir("/content/drive")[:10])
print("MyDrive exists:", os.path.exists("/content/drive/MyDrive"))
print("Project5 exists:", os.path.exists("/content/drive/MyDrive/MLP/Project5"))
print("dataset_expand exists:", os.path.exists("/content/drive/MyDrive/MLP/Project5/dataset_expand"))

if os.path.exists("/content/drive/MyDrive/MLP/Project5"):
    print(os.listdir("/content/drive/MyDrive/MLP/Project5"))

drive root: ['MyDrive', '.shortcut-targets-by-id', '.Trash-0', '.Encrypted']
MyDrive exists: True
Project5 exists: True
dataset_expand exists: True
['Project5_dino_vit_b16_baseline_expand_10ep.ipynb', 'dataset_expand', '期中资料', 'final_outputs', 'Project5_unified_training_evaluation_template.ipynb.ipynb', 'Project5_vit_b16_baseline_expand_10ep.ipynb']


# **00_config**
这一节集中管理本次 DINO-ViT-B/16 baseline 实验的所有配置，包括实验名称、模型名称、数据路径、训练参数、类别设置和输出路径。

当前实验使用 expanded dataset，并以 `test_original` 和 `test_jpeg_q30_controlled` 作为正式评估集。`test_jpeg_q30_controlled` 是重新生成的受控 JPEG q30 测试集，后续模型都应统一使用这一测试条件。

本次 DINO baseline 的核心目的是：在与 ViT-B/16 相近的 Transformer 架构下，对比 supervised ViT pretraining 和 self-supervised DINO pretraining 在 fake/real detection 与 JPEG robustness 上的差异。

本节中最重要的设置包括：
1. 将 `experiment_name` 改为 DINO-ViT-B/16 对应名称，避免覆盖 ViT 或 ResNet 结果；
2. 将 `model_name` 设置为 `dino_vit_b16`；
3. 训练 epoch 设置为 10；
4. batch size 先设置为 16，若 T4 爆显存再改为 8；
5. learning rate 设置为 1e-5，保证 full fine-tuning 相对稳定。

In [37]:
# ============================================================
# 00_config
# ============================================================

import os
import json

# ============================================================
# Drive mount safety check
# 重要：必须先成功 mount Google Drive，再运行本 cell。
# 如果 Drive 没挂载，或者 dataset_expand 路径不存在，本 cell 会直接停止，
# 避免误创建假的 /content/drive 本地目录。
# ============================================================

DRIVE_ROOT = "/content/drive/MyDrive"
PROJECT_ROOT = "/content/drive/MyDrive/MLP/Project5"
DATA_ROOT = "/content/drive/MyDrive/MLP/Project5/dataset_expand"

if not os.path.exists(DRIVE_ROOT):
    raise RuntimeError(
        "Google Drive is not properly mounted.\n"
        "Please run the following before 00_config:\n"
        "from google.colab import drive\n"
        "drive.mount('/content/drive')"
    )

if not os.path.exists(PROJECT_ROOT):
    raise RuntimeError(
        f"Project root does not exist: {PROJECT_ROOT}\n"
        "Please check whether you mounted the correct Google account."
    )

if not os.path.exists(DATA_ROOT):
    raise RuntimeError(
        f"Dataset path does not exist: {DATA_ROOT}\n"
        "Do NOT continue, because running config now may create fake local folders.\n"
        "Please check whether dataset_expand exists in the current Google Drive account/path."
    )

print("Drive mount check passed.")
print("Project root:", PROJECT_ROOT)
print("Data root:", DATA_ROOT)


# ============================================================
# Experiment config
# ============================================================

CONFIG = {
    # experiment
    "experiment_name": "dinov3_vit_b16_baseline_expand_10ep",
    "model_name": "dinov3_vit_b16",
    "method_name": "baseline",
    "dataset_version": "20pct_expand",

    # paths
    "data_root": DATA_ROOT,

    "train_dir": "train",
    "val_dir": "val",
    "test_orig_dir": "test_original",
    "test_jpeg_dir": "test_jpeg_q30_controlled",

    "eval_condition": "jpeg_q30_controlled",

    "output_root": os.path.join(PROJECT_ROOT, "final_outputs"),

    # data
    "image_size": 224,
    "batch_size": 16,
    "num_workers": 2,

    # training
    "num_epochs": 10,
    "learning_rate": 1e-5,
    "weight_decay": 1e-4,
    "optimizer": "AdamW",
    "checkpoint_metric": "val_f1",
    "seed": 42,

    # label
    "positive_class_name": "fake",
    "jpeg_quality": 30,
}


# ============================================================
# Auto-generate output paths
# ============================================================

CONFIG["output_dir"] = os.path.join(
    CONFIG["output_root"],
    CONFIG["experiment_name"]
)

os.makedirs(CONFIG["output_dir"], exist_ok=True)

CONFIG["checkpoint_path"] = os.path.join(
    CONFIG["output_dir"],
    "best_checkpoint.pt"
)

CONFIG["training_log_path"] = os.path.join(
    CONFIG["output_dir"],
    "training_log.csv"
)

CONFIG["summary_path"] = os.path.join(
    CONFIG["output_dir"],
    "summary.csv"
)


# ============================================================
# Save config for reproducibility
# ============================================================

with open(os.path.join(CONFIG["output_dir"], "config.json"), "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2, ensure_ascii=False)


# ============================================================
# Print config summary
# ============================================================

print("experiment_name:", CONFIG["experiment_name"])
print("model_name:", CONFIG["model_name"])
print("method_name:", CONFIG["method_name"])
print("dataset_version:", CONFIG["dataset_version"])

print("data_root:", CONFIG["data_root"])
print("train_dir:", CONFIG["train_dir"])
print("val_dir:", CONFIG["val_dir"])
print("test_orig_dir:", CONFIG["test_orig_dir"])
print("test_jpeg_dir:", CONFIG["test_jpeg_dir"])

print("output_dir:", CONFIG["output_dir"])
print("checkpoint_path:", CONFIG["checkpoint_path"])
print("training_log_path:", CONFIG["training_log_path"])
print("summary_path:", CONFIG["summary_path"])

print("batch_size:", CONFIG["batch_size"])
print("num_epochs:", CONFIG["num_epochs"])
print("learning_rate:", CONFIG["learning_rate"])
print("weight_decay:", CONFIG["weight_decay"])
print("positive_class_name:", CONFIG["positive_class_name"])

Drive mount check passed.
Project root: /content/drive/MyDrive/MLP/Project5
Data root: /content/drive/MyDrive/MLP/Project5/dataset_expand
experiment_name: dinov3_vit_b16_baseline_expand_10ep
model_name: dinov3_vit_b16
method_name: baseline
dataset_version: 20pct_expand
data_root: /content/drive/MyDrive/MLP/Project5/dataset_expand
train_dir: train
val_dir: val
test_orig_dir: test_original
test_jpeg_dir: test_jpeg_q30_controlled
output_dir: /content/drive/MyDrive/MLP/Project5/final_outputs/dinov3_vit_b16_baseline_expand_10ep
checkpoint_path: /content/drive/MyDrive/MLP/Project5/final_outputs/dinov3_vit_b16_baseline_expand_10ep/best_checkpoint.pt
training_log_path: /content/drive/MyDrive/MLP/Project5/final_outputs/dinov3_vit_b16_baseline_expand_10ep/training_log.csv
summary_path: /content/drive/MyDrive/MLP/Project5/final_outputs/dinov3_vit_b16_baseline_expand_10ep/summary.csv
batch_size: 16
num_epochs: 10
learning_rate: 1e-05
weight_decay: 0.0001
positive_class_name: fake


# **01_imports_and_seed**
这一节导入实验所需的 Python 库，并设置随机种子和运行设备。

DINO-ViT-B/16 使用 `timm` 加载预训练模型，因此需要确认当前环境已经安装并导入 timm。随机种子用于提高实验可复现性；device 会自动判断是否使用 GPU。

正式训练前建议确认 device 为 `cuda`，并使用 `nvidia-smi` 查看当前 GPU。不要使用 G4，优先使用 T4 或 L4。

In [38]:
# ============================================================
# 01_imports_and_seed
# ============================================================

# 如果当前环境没有 timm，先运行安装
try:
    import timm
except ImportError:
    !pip install timm -q
    import timm

import os
import json
import random
import time
import numpy as np
import pandas as pd

from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if torch.cuda.is_available():
    !nvidia-smi

Device: cuda
Mon Jun  1 15:27:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P0             34W /   70W |    3729MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+----------------------------------

# **02_dataset_and_dataloader**

这一节负责读取 expanded dataset，并构建 train、val、test_original 和 test_jpeg 四个 dataloader。

当前 DINO-ViT-B/16 baseline 继续使用与 ResNet18 和 ViT-B/16 相同的数据读取逻辑，以保证模型对比公平。正式 JPEG 测试集使用 `test_jpeg_q30_controlled`，不再使用原 expanded dataset 中退化过弱的 `test_jpeg`。

本节会完成：
1. 读取 `train` 和 `val`；
2. 读取 `test_original`；
3. 读取 `test_jpeg_q30_controlled`；
4. 统一将图片转成 RGB；
5. 使用 ImageNet normalization；
6. 从 train dataset 中读取 `class_to_idx`；
7. 设置 `fake` 为 positive class；
8. 打印各数据集大小，确认路径是否正确。

如果输出中的 `test_original` 和 `test_jpeg` 数量不一致，需要先暂停检查路径。

In [39]:
# ============================================================
# 02_dataset_and_dataloader
# ============================================================

def rgb_loader(path):
    try:
        with open(path, "rb") as f:
            img = Image.open(f)
            img.load()
            return img.convert("RGB")
    except Exception as e:
        print("Error loading image:", path)
        print("Error message:", e)
        raise e

train_transform = transforms.Compose([
    transforms.Resize((CONFIG["image_size"], CONFIG["image_size"])),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((CONFIG["image_size"], CONFIG["image_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_path = os.path.join(CONFIG["data_root"], CONFIG["train_dir"])
val_path = os.path.join(CONFIG["data_root"], CONFIG["val_dir"])
test_orig_path = os.path.join(CONFIG["data_root"], CONFIG["test_orig_dir"])
test_jpeg_path = os.path.join(CONFIG["data_root"], CONFIG["test_jpeg_dir"])

print("train_path:", train_path)
print("val_path:", val_path)
print("test_orig_path:", test_orig_path)
print("test_jpeg_path:", test_jpeg_path)

train_dataset = datasets.ImageFolder(
    train_path,
    transform=train_transform,
    loader=rgb_loader
)

val_dataset = datasets.ImageFolder(
    val_path,
    transform=eval_transform,
    loader=rgb_loader
)

test_orig_dataset = datasets.ImageFolder(
    test_orig_path,
    transform=eval_transform,
    loader=rgb_loader
)

test_jpeg_dataset = datasets.ImageFolder(
    test_jpeg_path,
    transform=eval_transform,
    loader=rgb_loader
)

class_to_idx = train_dataset.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}

print("class_to_idx:", class_to_idx)
print("idx_to_class:", idx_to_class)

positive_class_name = CONFIG["positive_class_name"]

if positive_class_name not in class_to_idx:
    raise ValueError(f"{positive_class_name} not found in class_to_idx: {class_to_idx}")

CONFIG["positive_label"] = class_to_idx[positive_class_name]

print("Positive class:", positive_class_name, "=>", CONFIG["positive_label"])

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"]
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"]
)

test_orig_loader = DataLoader(
    test_orig_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"]
)

test_jpeg_loader = DataLoader(
    test_jpeg_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"]
)

print("train:", len(train_dataset))
print("val:", len(val_dataset))
print("test_original:", len(test_orig_dataset))
print("test_jpeg:", len(test_jpeg_dataset))

print("\nOriginal examples:")
for p, y in test_orig_dataset.samples[:5]:
    print(os.path.basename(p), idx_to_class[y])

print("\nJPEG examples:")
for p, y in test_jpeg_dataset.samples[:5]:
    print(os.path.basename(p), idx_to_class[y])

train_path: /content/drive/MyDrive/MLP/Project5/dataset_expand/train
val_path: /content/drive/MyDrive/MLP/Project5/dataset_expand/val
test_orig_path: /content/drive/MyDrive/MLP/Project5/dataset_expand/test_original
test_jpeg_path: /content/drive/MyDrive/MLP/Project5/dataset_expand/test_jpeg_q30_controlled
class_to_idx: {'fake': 0, 'real': 1}
idx_to_class: {0: 'fake', 1: 'real'}
Positive class: fake => 0
train: 9996
val: 2004
test_original: 12005
test_jpeg: 12005

Original examples:
0001.jpg fake
0002.jpg fake
0003.jpg fake
0004.jpg fake
0005.jpg fake

JPEG examples:
0001.jpg fake
0002.jpg fake
0003.jpg fake
0004.jpg fake
0005.jpg fake


# **03_model_builder**
这一节负责根据 `CONFIG["model_name"]` 构建模型。

当前版本从 ViT-B/16 baseline 切换为 DINO-ViT-B/16 baseline。DINO baseline 使用 timm 中的 DINO 预训练 ViT 模型，并将分类头设置为 fake / real 二分类。

本实验采用 full fine-tuning，使 DINO baseline 与前面 ViT-B/16 baseline 的训练策略尽量保持一致。这样后续可以比较 supervised ViT pretraining 与 self-supervised DINO pretraining 在 fake image detection 和 JPEG robustness 上的差异。

## **03-0. 检查 timm 中可用的 DINO 模型**

In [21]:
# ============================================================
# 03-0_check_available_dino_models
# ============================================================

dino_models = [m for m in timm.list_models("*dino*") if "vit" in m.lower()]
print("Number of DINO-related ViT models:", len(dino_models))
dino_models[:50]

Number of DINO-related ViT models: 19


['vit_7b_patch16_dinov3',
 'vit_base_patch14_dinov2',
 'vit_base_patch14_reg4_dinov2',
 'vit_base_patch16_dinov3',
 'vit_base_patch16_dinov3_qkvb',
 'vit_giant_patch14_dinov2',
 'vit_giant_patch14_reg4_dinov2',
 'vit_huge_plus_patch16_dinov3',
 'vit_huge_plus_patch16_dinov3_qkvb',
 'vit_large_patch14_dinov2',
 'vit_large_patch14_reg4_dinov2',
 'vit_large_patch16_dinov3',
 'vit_large_patch16_dinov3_qkvb',
 'vit_small_patch14_dinov2',
 'vit_small_patch14_reg4_dinov2',
 'vit_small_patch16_dinov3',
 'vit_small_patch16_dinov3_qkvb',
 'vit_small_plus_patch16_dinov3',
 'vit_small_plus_patch16_dinov3_qkvb']

## **03-1. 构建模型函数**
这一小节定义统一的模型构建函数。

当前函数保留 ResNet18 和 ViT-B/16 分支，并新增 DINO-ViT-B/16 分支。当 `model_name = "dino_vit_b16"` 时，会加载 timm 的 DINO 预训练 ViT-B/16 模型，并将输出类别数设置为 2。

如果运行时报错说明模型名不存在，需要先检查上一小节的 DINO 模型列表，并替换为当前环境中可用的 DINO 模型名。

In [27]:
# ============================================================
# 03-1_model_builder
# ============================================================

def build_model(model_name, num_classes=2):
    if model_name == "resnet18":
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
        return model

    elif model_name == "vit_b16":
        model = timm.create_model(
            "vit_base_patch16_224",
            pretrained=True,
            num_classes=num_classes
        )
        return model

    elif model_name == "dinov3_vit_b16":
        model = timm.create_model(
            "vit_base_patch16_dinov3",
            pretrained=True,
            num_classes=num_classes
        )
        return model

    else:
        raise ValueError(f"Unsupported model_name: {model_name}")

## **03-2. 初始化 model / loss / optimizer**
这一小节根据 config 构建 DINO-ViT-B/16 模型，并初始化 loss function 和 optimizer。

为了和 ViT-B/16 baseline 保持可比性，DINO baseline 第一版采用 full fine-tuning，使用 CrossEntropyLoss 和 AdamW。learning rate 先保持 1e-5，避免 full fine-tuning 过于不稳定。

In [40]:
# ============================================================
# 03-2_initialize_model_loss_optimizer
# ============================================================

model = build_model(CONFIG["model_name"], num_classes=len(class_to_idx))
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"]
)

print("Model:", CONFIG["model_name"])
print("Number of classes:", len(class_to_idx))
print("Class mapping:", class_to_idx)
print("Positive label:", CONFIG["positive_label"], CONFIG["positive_class_name"])
print("Optimizer:", CONFIG["optimizer"])
print("Learning rate:", CONFIG["learning_rate"])
print("Weight decay:", CONFIG["weight_decay"])

Model: dinov3_vit_b16
Number of classes: 2
Class mapping: {'fake': 0, 'real': 1}
Positive label: 0 fake
Optimizer: AdamW
Learning rate: 1e-05
Weight decay: 0.0001


## **03-3. 检查是不是 full fine-tuning**
这一小节用于检查 DINO-ViT-B/16 中哪些参数是可训练的。

当前 DINO baseline 采用 full fine-tuning，即整个 DINO-ViT 模型都会参与训练。这样可以和 ViT-B/16 full fine-tuning baseline 保持一致。如果后续改成 frozen backbone + classifier head，需要在 experiment name、config 和报告中明确标注。

In [42]:
# ============================================================
# 03-3_check_trainable_parameters
# ============================================================

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print("Trainable ratio:", trainable_params / total_params)

print("\nFirst 20 parameter groups:")
for i, (name, param) in enumerate(model.named_parameters()):
    print(name, param.requires_grad, tuple(param.shape))
    if i >= 19:
        break

Total parameters: 85642754
Trainable parameters: 85642754
Trainable ratio: 1.0

First 20 parameter groups:
cls_token True (1, 1, 768)
reg_token True (1, 4, 768)
patch_embed.proj.weight True (768, 3, 16, 16)
patch_embed.proj.bias True (768,)
blocks.0.gamma_1 True (768,)
blocks.0.gamma_2 True (768,)
blocks.0.norm1.weight True (768,)
blocks.0.norm1.bias True (768,)
blocks.0.attn.qkv.weight True (2304, 768)
blocks.0.attn.proj.weight True (768, 768)
blocks.0.attn.proj.bias True (768,)
blocks.0.norm2.weight True (768,)
blocks.0.norm2.bias True (768,)
blocks.0.mlp.fc1.weight True (3072, 768)
blocks.0.mlp.fc1.bias True (3072,)
blocks.0.mlp.fc2.weight True (768, 3072)
blocks.0.mlp.fc2.bias True (768,)
blocks.1.gamma_1 True (768,)
blocks.1.gamma_2 True (768,)
blocks.1.norm1.weight True (768,)


## **03-4. 跑一个 mini batch 测试**
这一小节用一个 mini batch 检查 DINO-ViT-B/16 是否能正常前向传播，并确认输出维度是否为 `[batch_size, 2]`。

如果输出是 `[batch_size, 2]`，说明分类头已经正确设置为 fake / real 二分类，可以继续正式训练。如果输出是 `[batch_size, 1000]` 或其他格式，需要先检查模型分类头设置。

In [43]:
# ============================================================
# 03-4_mini_batch_forward_test
# ============================================================

model.eval()

images, labels = next(iter(train_loader))
images = images.to(device)
labels = labels.to(device)

with torch.no_grad():
    outputs = model(images)
    preds = torch.argmax(outputs, dim=1)

print("Input images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Outputs shape:", outputs.shape)
print("Preds shape:", preds.shape)
print("First 5 labels:", labels[:5].detach().cpu().numpy())
print("First 5 preds:", preds[:5].detach().cpu().numpy())

if outputs.shape[1] != len(class_to_idx):
    raise ValueError(f"Output dim {outputs.shape[1]} != num_classes {len(class_to_idx)}")

Input images shape: torch.Size([16, 3, 224, 224])
Labels shape: torch.Size([16])
Outputs shape: torch.Size([16, 2])
Preds shape: torch.Size([16])
First 5 labels: [1 0 1 1 1]
First 5 preds: [0 0 0 0 0]


# **04🟢train_one_epoch**
这一节定义单个 epoch 的训练逻辑，包括前向传播、loss 计算、反向传播、参数更新，以及 train loss / train accuracy 统计。

这一节只是定义函数，本身不会开始训练，所以运行后没有明显输出是正常的。真正开始训练的是 `07_train_and_save_best_checkpoint`。

In [44]:
# ============================================================
# 04_train_one_epoch
# ============================================================

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    all_labels = []
    all_preds = []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        batch_size = images.size(0)
        total_loss += loss.item() * batch_size

        preds = torch.argmax(outputs, dim=1)

        all_labels.extend(labels.detach().cpu().numpy())
        all_preds.extend(preds.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)

    return {
        "train_loss": avg_loss,
        "train_acc": acc,
    }

print("train_one_epoch function is ready.")

train_one_epoch function is ready.


# **05_evaluate_function**
这一节定义统一评估函数，用于 validation、test_original 和 test_jpeg_q30_controlled。

为了保证 ResNet18、ViT-B/16、DINO-ViT-B/16 和后续 LazyStrike 结果可比较，所有模型都应使用同一个 evaluation function。

本项目重点关注 fake 类的 recall，以及 original 到 JPEG q30 controlled 的 recall drop。

In [45]:
# ============================================================
# 05_evaluate_function
# ============================================================

def evaluate_model(model, loader, criterion, device, pos_label):
    model.eval()

    total_loss = 0.0
    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)

            batch_size = images.size(0)
            total_loss += loss.item() * batch_size

            all_labels.extend(labels.detach().cpu().numpy())
            all_preds.extend(preds.detach().cpu().numpy())
            all_probs.extend(probs.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)

    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, pos_label=pos_label, zero_division=0)
    recall = recall_score(all_labels, all_preds, pos_label=pos_label, zero_division=0)
    f1 = f1_score(all_labels, all_preds, pos_label=pos_label, zero_division=0)
    cm = confusion_matrix(all_labels, all_preds)

    return {
        "loss": avg_loss,
        "accuracy": acc,
        "precision_fake": precision,
        "recall_fake": recall,
        "f1_fake": f1,
        "confusion_matrix": cm.tolist(),
        "labels": all_labels,
        "preds": all_preds,
        "probs": np.array(all_probs),
    }

print("evaluate_model function is ready.")

evaluate_model function is ready.


在正式训练前，先对 val 跑一次未训练模型的评估，确认函数没问题

In [47]:
val_result_before_train = evaluate_model(
    model=model,
    loader=val_loader,
    criterion=criterion,
    device=device,
    pos_label=CONFIG["positive_label"]
)

print("Val loss:", val_result_before_train["loss"])
print("Val acc:", val_result_before_train["accuracy"])
print("Val precision_fake:", val_result_before_train["precision_fake"])
print("Val recall_fake:", val_result_before_train["recall_fake"])
print("Val f1_fake:", val_result_before_train["f1_fake"])
print("Confusion matrix:")
print(val_result_before_train["confusion_matrix"])

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (161087488 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (96000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Val loss: 0.6931623504785245
Val acc: 0.48353293413173654
Val precision_fake: 0.48995739500912966
Val recall_fake: 0.8033932135728543
Val f1_fake: 0.6086956521739131
Confusion matrix:
[[805, 197], [838, 164]]


# **06_save_predictions_csv**
这一节定义用于保存逐图像预测结果的函数，本身通常不会产生输出。

真正生成 CSV 文件的动作发生在 `08_test_original_and_jpeg`，因为 08 会在测试完成后调用这里定义的保存函数。

生成的 predictions CSV 后续可用于 original / JPEG paired analysis、TP→FN case 查找，以及 Grad-CAM 或 attention visualization 样本选择。

In [48]:
# ============================================================
# 06_save_predictions_csv
# ============================================================

def make_image_id(path):
    filename = os.path.basename(path)
    stem = os.path.splitext(filename)[0]
    return stem

def save_predictions_csv(dataset, eval_result, split_name, config, idx_to_class, filename=None):
    rows = []

    labels = eval_result["labels"]
    preds = eval_result["preds"]
    probs = eval_result["probs"]

    for i, (path, _) in enumerate(dataset.samples):
        true_label = int(labels[i])
        pred_label = int(preds[i])
        prob_values = probs[i]

        row = {
            "image_path": path,
            "filename": os.path.basename(path),
            "image_id": make_image_id(path),

            "true_label_idx": true_label,
            "true_label_name": idx_to_class[true_label],

            "pred_label_idx": pred_label,
            "pred_label_name": idx_to_class[pred_label],

            "correct": int(true_label == pred_label),
            "split": split_name,

            "model_name": config["model_name"],
            "method_name": config["method_name"],
            "experiment_name": config["experiment_name"],
        }

        for class_idx, class_name in idx_to_class.items():
            row[f"prob_{class_name}"] = float(prob_values[class_idx])

        rows.append(row)

    df = pd.DataFrame(rows)

    if filename is None:
        filename = f"predictions_{split_name}.csv"

    save_path = os.path.join(
        config["output_dir"],
        filename
    )

    df.to_csv(save_path, index=False, encoding="utf-8-sig")
    print(f"Saved predictions to: {save_path}")

    return df

def save_metrics_json(result, save_path):
    clean_result = {
        "loss": float(result["loss"]),
        "accuracy": float(result["accuracy"]),
        "precision_fake": float(result["precision_fake"]),
        "recall_fake": float(result["recall_fake"]),
        "f1_fake": float(result["f1_fake"]),
        "confusion_matrix": result["confusion_matrix"],
    }

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(clean_result, f, indent=2, ensure_ascii=False)

    print(f"Saved metrics to: {save_path}")

print("save_predictions_csv and save_metrics_json functions are ready.")

save_predictions_csv and save_metrics_json functions are ready.


# **07🟢train_and_save_best_checkpoint**
这一节是正式训练循环。

当前 DINO-ViT-B/16 baseline 将最大训练轮数设置为 10 epoch。训练会跑满 10 epoch，并在每个 epoch 后使用 validation set 评估模型表现。最终保存 validation fake F1 最高的 checkpoint，而不是直接使用最后一个 epoch。

本节已加入每个 epoch 的训练耗时统计，并将 `epoch_time_sec` 和 `epoch_time_min` 写入 `training_log.csv`，方便后续比较不同模型的训练成本。

In [49]:
# ============================================================
# 07_train_and_save_best_checkpoint
# ============================================================

best_val_f1 = -1.0
best_epoch = -1
training_logs = []

total_train_start = time.time()

for epoch in range(1, CONFIG["num_epochs"] + 1):
    epoch_start = time.time()

    train_metrics = train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device
    )

    val_result = evaluate_model(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=device,
        pos_label=CONFIG["positive_label"]
    )

    epoch_time_sec = time.time() - epoch_start
    epoch_time_min = epoch_time_sec / 60

    log_row = {
        "epoch": epoch,

        "train_loss": train_metrics["train_loss"],
        "train_acc": train_metrics["train_acc"],

        "val_loss": val_result["loss"],
        "val_acc": val_result["accuracy"],
        "val_precision_fake": val_result["precision_fake"],
        "val_recall_fake": val_result["recall_fake"],
        "val_f1_fake": val_result["f1_fake"],

        "epoch_time_sec": epoch_time_sec,
        "epoch_time_min": epoch_time_min,
    }

    training_logs.append(log_row)

    print(
        f"Epoch {epoch}/{CONFIG['num_epochs']} | "
        f"train_loss={train_metrics['train_loss']:.4f}, "
        f"train_acc={train_metrics['train_acc']:.4f}, "
        f"val_loss={val_result['loss']:.4f}, "
        f"val_acc={val_result['accuracy']:.4f}, "
        f"val_precision_fake={val_result['precision_fake']:.4f}, "
        f"val_recall_fake={val_result['recall_fake']:.4f}, "
        f"val_f1_fake={val_result['f1_fake']:.4f}, "
        f"epoch_time={epoch_time_min:.2f} min"
    )

    current_val_f1 = val_result["f1_fake"]

    if current_val_f1 > best_val_f1:
        best_val_f1 = current_val_f1
        best_epoch = epoch

        checkpoint = {
            "epoch": epoch,
            "best_epoch": best_epoch,
            "best_val_f1": best_val_f1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "config": CONFIG,
            "class_to_idx": class_to_idx,
            "idx_to_class": idx_to_class,
        }

        torch.save(checkpoint, CONFIG["checkpoint_path"])
        print(f"Saved best checkpoint at epoch {epoch}, val_f1={best_val_f1:.4f}")

total_train_time_sec = time.time() - total_train_start
total_train_time_min = total_train_time_sec / 60
total_train_time_hour = total_train_time_min / 60

training_log_df = pd.DataFrame(training_logs)
training_log_df.to_csv(CONFIG["training_log_path"], index=False, encoding="utf-8-sig")

print("Training finished.")
print("Best epoch:", best_epoch)
print("Best val F1:", best_val_f1)
print("Total training time:", f"{total_train_time_sec:.2f} sec / {total_train_time_min:.2f} min / {total_train_time_hour:.2f} h")
print("Training log saved to:", CONFIG["training_log_path"])
print("Best checkpoint saved to:", CONFIG["checkpoint_path"])

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (96012000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (90671520 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (107184040 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-p

Epoch 1/10 | train_loss=0.3138, train_acc=0.9102, val_loss=0.1325, val_acc=0.9516, val_precision_fake=0.9840, val_recall_fake=0.9182, val_f1_fake=0.9499, epoch_time=37.03 min
Saved best checkpoint at epoch 1, val_f1=0.9499


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (107184040 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (90671520 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (96012000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-p

Epoch 2/10 | train_loss=0.0860, train_acc=0.9737, val_loss=0.0886, val_acc=0.9671, val_precision_fake=0.9885, val_recall_fake=0.9451, val_f1_fake=0.9663, epoch_time=20.59 min
Saved best checkpoint at epoch 2, val_f1=0.9663


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (96012000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (90671520 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (98058240 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (107184040 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/us

...

Epoch 8/10 | train_loss=0.0068, train_acc=0.9985, val_loss=0.1427, val_acc=0.9581, val_precision_fake=0.9371, val_recall_fake=0.9820, val_f1_fake=0.9591, epoch_time=20.56 min


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (107184040 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (90671520 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (96012000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-p

Epoch 9/10 | train_loss=0.0138, train_acc=0.9958, val_loss=0.1085, val_acc=0.9696, val_precision_fake=0.9968, val_recall_fake=0.9421, val_f1_fake=0.9687, epoch_time=20.20 min


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (98058240 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (107184040 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (90671520 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (96012000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/us

Epoch 10/10 | train_loss=0.0188, train_acc=0.9930, val_loss=0.0821, val_acc=0.9770, val_precision_fake=0.9780, val_recall_fake=0.9760, val_f1_fake=0.9770, epoch_time=20.33 min
Saved best checkpoint at epoch 10, val_f1=0.9770
Training finished.
Best epoch: 10
Best val F1: 0.977022977022977
Total training time: 13355.26 sec / 222.59 min / 3.71 h
Training log saved to: /content/drive/MyDrive/MLP/Project5/final_outputs/dinov3_vit_b16_baseline_expand_10ep/training_log.csv
Best checkpoint saved to: /content/drive/MyDrive/MLP/Project5/final_outputs/dinov3_vit_b16_baseline_expand_10ep/best_checkpoint.pt


# **08_test_original_and_jpeg**

这一节加载 DINO-ViT-B/16 的 best checkpoint，并分别在 `test_original` 和 `test_jpeg_q30_controlled` 上进行最终测试。

本节已经改成分阶段保存：
1. 先测试 original；
2. original 测完后立刻保存 `metrics_original.json` 和 `predictions_original.csv`；
3. 再测试 JPEG q30 controlled；
4. JPEG 测完后立刻保存 `metrics_jpeg.json` 和 `predictions_jpeg.csv`。

这样即使中途断线，已经跑完的 original 结果也不会丢失。

In [50]:
# ============================================================
# 08_test_original_and_jpeg
# ============================================================

checkpoint = torch.load(CONFIG["checkpoint_path"], map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)

best_epoch = checkpoint.get("best_epoch", checkpoint.get("epoch", None))
best_val_f1 = checkpoint.get("best_val_f1", None)

print("Loaded best checkpoint:")
print("Best epoch:", best_epoch)
print("Best val F1:", best_val_f1)

# ------------------------------------------------------------
# 1. Evaluate and save ORIGINAL first
# ------------------------------------------------------------

orig_start = time.time()

orig_result = evaluate_model(
    model=model,
    loader=test_orig_loader,
    criterion=criterion,
    device=device,
    pos_label=CONFIG["positive_label"]
)

orig_time_sec = time.time() - orig_start
orig_time_min = orig_time_sec / 60

print("\nOriginal Test Result")
print("Accuracy:", orig_result["accuracy"])
print("Precision_fake:", orig_result["precision_fake"])
print("Recall_fake:", orig_result["recall_fake"])
print("F1_fake:", orig_result["f1_fake"])
print("Confusion Matrix:")
print(orig_result["confusion_matrix"])
print(f"Original evaluation time: {orig_time_min:.2f} min")

save_metrics_json(
    orig_result,
    os.path.join(CONFIG["output_dir"], "metrics_original.json")
)

pred_orig_df = save_predictions_csv(
    dataset=test_orig_dataset,
    eval_result=orig_result,
    split_name="original",
    config=CONFIG,
    idx_to_class=idx_to_class,
    filename="predictions_original.csv"
)

print("Original test metrics and predictions saved.")

# ------------------------------------------------------------
# 2. Evaluate and save JPEG q30 controlled
# ------------------------------------------------------------

jpeg_start = time.time()

jpeg_result = evaluate_model(
    model=model,
    loader=test_jpeg_loader,
    criterion=criterion,
    device=device,
    pos_label=CONFIG["positive_label"]
)

jpeg_time_sec = time.time() - jpeg_start
jpeg_time_min = jpeg_time_sec / 60

print("\nJPEG Test Result")
print("Accuracy:", jpeg_result["accuracy"])
print("Precision_fake:", jpeg_result["precision_fake"])
print("Recall_fake:", jpeg_result["recall_fake"])
print("F1_fake:", jpeg_result["f1_fake"])
print("Confusion Matrix:")
print(jpeg_result["confusion_matrix"])
print(f"JPEG evaluation time: {jpeg_time_min:.2f} min")

save_metrics_json(
    jpeg_result,
    os.path.join(CONFIG["output_dir"], "metrics_jpeg.json")
)

pred_jpeg_df = save_predictions_csv(
    dataset=test_jpeg_dataset,
    eval_result=jpeg_result,
    split_name="jpeg",
    config=CONFIG,
    idx_to_class=idx_to_class,
    filename="predictions_jpeg.csv"
)

print("JPEG test metrics and predictions saved.")

Loaded best checkpoint:
Best epoch: 10
Best val F1: 0.977022977022977


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (143040000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (121554000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(



Original Test Result
Accuracy: 0.9746772178259059
Precision_fake: 0.9748249416472158
Recall_fake: 0.9745
F1_fake: 0.9746624437406234
Confusion Matrix:
[[5847, 153], [151, 5854]]
Original evaluation time: 87.36 min
Saved metrics to: /content/drive/MyDrive/MLP/Project5/final_outputs/dinov3_vit_b16_baseline_expand_10ep/metrics_original.json
Saved predictions to: /content/drive/MyDrive/MLP/Project5/final_outputs/dinov3_vit_b16_baseline_expand_10ep/predictions_original.csv
Original test metrics and predictions saved.


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (143040000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (121554000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(



JPEG Test Result
Accuracy: 0.9595168679716785
Precision_fake: 0.9848751319029194
Recall_fake: 0.9333333333333333
F1_fake: 0.9584117747732329
Confusion Matrix:
[[5600, 400], [86, 5919]]
JPEG evaluation time: 92.16 min
Saved metrics to: /content/drive/MyDrive/MLP/Project5/final_outputs/dinov3_vit_b16_baseline_expand_10ep/metrics_jpeg.json
Saved predictions to: /content/drive/MyDrive/MLP/Project5/final_outputs/dinov3_vit_b16_baseline_expand_10ep/predictions_jpeg.csv
JPEG test metrics and predictions saved.


# **09_generate_summary_table**
这一节将 DINO-ViT-B/16 在 original test 和 JPEG q30 controlled test 上的结果汇总为一行 summary。

summary 中会包含 original / JPEG 的各项指标，以及 accuracy drop、recall drop 和 F1 drop。后续可以将 ResNet18、ViT-B/16、DINO-ViT-B/16 和 LazyStrike 的 summary 合并成最终模型对比表。

In [1]:
# ============================================================
# 09_generate_summary_table_from_saved_json
# 用于 Colab 断线后，从已保存 metrics_original.json / metrics_jpeg.json
# 重新生成 summary.csv
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import json
import pandas as pd
import torch

# 手动恢复本次实验的基本配置
CONFIG = {
    "experiment_name": "dinov3_vit_b16_baseline_expand_10ep",
    "model_name": "dinov3_vit_b16",
    "method_name": "baseline",
    "dataset_version": "20pct_expand",
    "eval_condition": "jpeg_q30_controlled",

    "output_dir": "/content/drive/MyDrive/MLP/Project5/final_outputs/dinov3_vit_b16_baseline_expand_10ep",

    "batch_size": 16,
    "num_epochs": 10,
    "learning_rate": 1e-5,
    "weight_decay": 1e-4,
}

checkpoint_path = os.path.join(CONFIG["output_dir"], "best_checkpoint.pt")
metrics_original_path = os.path.join(CONFIG["output_dir"], "metrics_original.json")
metrics_jpeg_path = os.path.join(CONFIG["output_dir"], "metrics_jpeg.json")
summary_path = os.path.join(CONFIG["output_dir"], "summary.csv")

# 读取 checkpoint，拿 best_epoch / best_val_f1
checkpoint = torch.load(checkpoint_path, map_location="cpu")
best_epoch = checkpoint.get("best_epoch", checkpoint.get("epoch", None))
best_val_f1 = checkpoint.get("best_val_f1", None)

# 读取已经保存好的 metrics
with open(metrics_original_path, "r", encoding="utf-8") as f:
    orig_result = json.load(f)

with open(metrics_jpeg_path, "r", encoding="utf-8") as f:
    jpeg_result = json.load(f)

summary = {
    "experiment_name": CONFIG["experiment_name"],
    "model_name": CONFIG["model_name"],
    "method_name": CONFIG["method_name"],
    "dataset_version": CONFIG["dataset_version"],
    "eval_condition": CONFIG["eval_condition"],

    "best_epoch": best_epoch,
    "best_val_f1": best_val_f1,

    "batch_size": CONFIG["batch_size"],
    "num_epochs": CONFIG["num_epochs"],
    "learning_rate": CONFIG["learning_rate"],
    "weight_decay": CONFIG["weight_decay"],

    "orig_loss": orig_result["loss"],
    "orig_acc": orig_result["accuracy"],
    "orig_precision_fake": orig_result["precision_fake"],
    "orig_recall_fake": orig_result["recall_fake"],
    "orig_f1_fake": orig_result["f1_fake"],

    "jpeg_loss": jpeg_result["loss"],
    "jpeg_acc": jpeg_result["accuracy"],
    "jpeg_precision_fake": jpeg_result["precision_fake"],
    "jpeg_recall_fake": jpeg_result["recall_fake"],
    "jpeg_f1_fake": jpeg_result["f1_fake"],

    "acc_drop": orig_result["accuracy"] - jpeg_result["accuracy"],
    "recall_drop": orig_result["recall_fake"] - jpeg_result["recall_fake"],
    "f1_drop": orig_result["f1_fake"] - jpeg_result["f1_fake"],
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("Saved summary to:", summary_path)
summary_df

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved summary to: /content/drive/MyDrive/MLP/Project5/final_outputs/dinov3_vit_b16_baseline_expand_10ep/summary.csv


,experiment_name,model_name,method_name,dataset_version,eval_condition,best_epoch,best_val_f1,batch_size,num_epochs,learning_rate,...,orig_recall_fake,orig_f1_fake,jpeg_loss,jpeg_acc,jpeg_precision_fake,jpeg_recall_fake,jpeg_f1_fake,acc_drop,recall_drop,f1_drop
0,dinov3_vit_b16_baseline_expand_10ep,dinov3_vit_b16,baseline,20pct_expand,jpeg_q30_controlled,10,0.977023,16,10,0.00001,...,0.9745,0.974662,0.149126,0.959517,0.984875,0.933333,0.958412,0.01516,0.041167,0.016251


跑完 09 后归档结果

In [2]:
# ============================================================
# Archive DINOv3 evaluation outputs
# ============================================================

import os
import shutil

output_dir = "/content/drive/MyDrive/MLP/Project5/final_outputs/dinov3_vit_b16_baseline_expand_10ep"

archive_dir = os.path.join(output_dir, "eval_jpeg_q30_controlled")
os.makedirs(archive_dir, exist_ok=True)

rename_map = {
    "metrics_original.json": "metrics_original.json",
    "metrics_jpeg.json": "metrics_jpeg_q30_controlled.json",
    "predictions_original.csv": "predictions_original.csv",
    "predictions_jpeg.csv": "predictions_jpeg_q30_controlled.csv",
    "summary.csv": "summary_jpeg_q30_controlled.csv",
}

for src_name, dst_name in rename_map.items():
    src = os.path.join(output_dir, src_name)
    dst = os.path.join(archive_dir, dst_name)

    if os.path.exists(src):
        shutil.move(src, dst)
        print("Moved:", src, "->", dst)
    else:
        print("Not found:", src)

print("Done archiving DINOv3 controlled q30 evaluation.")

Moved: /content/drive/MyDrive/MLP/Project5/final_outputs/dinov3_vit_b16_baseline_expand_10ep/metrics_original.json -> /content/drive/MyDrive/MLP/Project5/final_outputs/dinov3_vit_b16_baseline_expand_10ep/eval_jpeg_q30_controlled/metrics_original.json
Moved: /content/drive/MyDrive/MLP/Project5/final_outputs/dinov3_vit_b16_baseline_expand_10ep/metrics_jpeg.json -> /content/drive/MyDrive/MLP/Project5/final_outputs/dinov3_vit_b16_baseline_expand_10ep/eval_jpeg_q30_controlled/metrics_jpeg_q30_controlled.json
Moved: /content/drive/MyDrive/MLP/Project5/final_outputs/dinov3_vit_b16_baseline_expand_10ep/predictions_original.csv -> /content/drive/MyDrive/MLP/Project5/final_outputs/dinov3_vit_b16_baseline_expand_10ep/eval_jpeg_q30_controlled/predictions_original.csv
Moved: /content/drive/MyDrive/MLP/Project5/final_outputs/dinov3_vit_b16_baseline_expand_10ep/predictions_jpeg.csv -> /content/drive/MyDrive/MLP/Project5/final_outputs/dinov3_vit_b16_baseline_expand_10ep/eval_jpeg_q30_controlled/predic

# **[草稿]test_jpeg_q30 验证+检查+重做**
这一部分是数据质量检查和修正记录，不属于正式训练流程。

这里记录了本次实验中对 JPEG 测试集的排查过程：

① 发现 expanded dataset 原本的 test_jpeg 与期中 q30 数据退化强度不一致。

② 通过文件大小、size ratio、PSNR 和 mean absolute difference 检查 JPEG 退化强度。

③ 确认期中旧 dataset 的 q30 是明显压缩，而 expanded 原 test_jpeg 更像 mild JPEG re-encoding。

④ 将 original 和 JPEG 测试集重新整理成独立文件夹，避免混读。

⑤ 基于 test_original 重新生成 test_jpeg_q30_controlled。

⑥ 使用 controlled q30 重新评估 ResNet18 后，JPEG-induced fake recall drop 重新出现。

这部分可以作为交接说明保留，但后续训练 ViT / DINO 时不需要重复执行。
后续正式实验统一使用 test_jpeg_q30_controlled。

In [ ]:
import os
import shutil

output_dir = "/content/drive/MyDrive/MLP/Project5/final_outputs/resnet18_baseline_expand"

archive_dir = os.path.join(output_dir, "eval_jpeg_q30_controlled")
os.makedirs(archive_dir, exist_ok=True)

rename_map = {
    "metrics_original.json": "metrics_original.json",
    "metrics_jpeg.json": "metrics_jpeg_q30_controlled.json",
    "predictions_original.csv": "predictions_original.csv",
    "predictions_jpeg.csv": "predictions_jpeg_q30_controlled.csv",
    "summary.csv": "summary_jpeg_q30_controlled.csv",
}

for src_name, dst_name in rename_map.items():
    src = os.path.join(output_dir, src_name)
    dst = os.path.join(archive_dir, dst_name)

    if os.path.exists(src):
        shutil.move(src, dst)
        print("Moved:", src, "->", dst)
    else:
        print("Not found:", src)

print("Done archiving controlled q30 evaluation.")

Moved: /content/drive/MyDrive/MLP/Project5/final_outputs/resnet18_baseline_expand/metrics_original.json -> /content/drive/MyDrive/MLP/Project5/final_outputs/resnet18_baseline_expand/eval_jpeg_q30_controlled/metrics_original.json
Moved: /content/drive/MyDrive/MLP/Project5/final_outputs/resnet18_baseline_expand/metrics_jpeg.json -> /content/drive/MyDrive/MLP/Project5/final_outputs/resnet18_baseline_expand/eval_jpeg_q30_controlled/metrics_jpeg_q30_controlled.json
Moved: /content/drive/MyDrive/MLP/Project5/final_outputs/resnet18_baseline_expand/predictions_original.csv -> /content/drive/MyDrive/MLP/Project5/final_outputs/resnet18_baseline_expand/eval_jpeg_q30_controlled/predictions_original.csv
Moved: /content/drive/MyDrive/MLP/Project5/final_outputs/resnet18_baseline_expand/predictions_jpeg.csv -> /content/drive/MyDrive/MLP/Project5/final_outputs/resnet18_baseline_expand/eval_jpeg_q30_controlled/predictions_jpeg_q30_controlled.csv
Moved: /content/drive/MyDrive/MLP/Project5/final_outputs/r

In [ ]:
# ============================================================
# Rebuild test original / jpeg dataloaders from mixed test folder
# ============================================================

import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader

class FilteredImageFolder(Dataset):
    def __init__(self, root, transform=None, mode="original", class_to_idx=None):
        """
        root: /content/drive/MyDrive/MLP/Project5/dataset_expand/test
        mode:
          - original: filenames WITHOUT "_JPEG"
          - jpeg: filenames WITH "_JPEG"
        """
        self.root = root
        self.transform = transform
        self.mode = mode

        if class_to_idx is None:
            classes = sorted([
                d for d in os.listdir(root)
                if os.path.isdir(os.path.join(root, d))
            ])
            self.class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}
        else:
            self.class_to_idx = class_to_idx

        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}
        self.samples = []

        valid_exts = (".jpg", ".jpeg", ".png", ".webp")

        for class_name, label in self.class_to_idx.items():
            class_dir = os.path.join(root, class_name)

            for fname in sorted(os.listdir(class_dir)):
                if not fname.lower().endswith(valid_exts):
                    continue

                is_jpeg_version = "_JPEG" in fname

                if mode == "original" and is_jpeg_version:
                    continue

                if mode == "jpeg" and not is_jpeg_version:
                    continue

                self.samples.append((os.path.join(class_dir, fname), label))

        print(f"[{mode}] total samples:", len(self.samples))

        # 按类别统计一下
        counts = {}
        for _, label in self.samples:
            cls = self.idx_to_class[label]
            counts[cls] = counts.get(cls, 0) + 1
        print(f"[{mode}] class counts:", counts)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        with open(path, "rb") as f:
            img = Image.open(f).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label


test_root = os.path.join(CONFIG["data_root"], "test")

test_orig_dataset = FilteredImageFolder(
    root=test_root,
    transform=eval_transform,
    mode="original",
    class_to_idx=class_to_idx
)

test_jpeg_dataset = FilteredImageFolder(
    root=test_root,
    transform=eval_transform,
    mode="jpeg",
    class_to_idx=class_to_idx
)

test_orig_loader = DataLoader(
    test_orig_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"]
)

test_jpeg_loader = DataLoader(
    test_jpeg_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"]
)

print("\nOriginal examples:")
for p, y in test_orig_dataset.samples[:5]:
    print(os.path.basename(p), idx_to_class[y])

print("\nJPEG examples:")
for p, y in test_jpeg_dataset.samples[:5]:
    print(os.path.basename(p), idx_to_class[y])

[original] total samples: 12008
[original] class counts: {'fake': 6000, 'real': 6008}
[jpeg] total samples: 12005
[jpeg] class counts: {'fake': 6000, 'real': 6005}

Original examples:
0001.jpg fake
0002.jpg fake
0003.jpg fake
0004.jpg fake
0005.jpg fake

JPEG examples:
0001_JPEG.jpg fake
0002_JPEG.jpg fake
0003_JPEG.jpg fake
0004_JPEG.jpg fake
0005_JPEG.jpg fake


In [ ]:
import os
import shutil
from tqdm import tqdm

src_test_root = "/content/drive/MyDrive/MLP/Project5/dataset_expand/test"

dst_orig_root = "/content/drive/MyDrive/MLP/Project5/dataset_expand/test_original"
dst_jpeg_root = "/content/drive/MyDrive/MLP/Project5/dataset_expand/test_jpeg"

classes = ["fake", "real"]

for cls in classes:
    src_dir = os.path.join(src_test_root, cls)
    dst_orig_dir = os.path.join(dst_orig_root, cls)
    dst_jpeg_dir = os.path.join(dst_jpeg_root, cls)

    os.makedirs(dst_orig_dir, exist_ok=True)
    os.makedirs(dst_jpeg_dir, exist_ok=True)

    files = [
        f for f in os.listdir(src_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
    ]

    for fname in tqdm(files, desc=f"Splitting {cls}"):
        src_path = os.path.join(src_dir, fname)

        if "_JPEG" in fname:
            # 复制到 test_jpeg，并把 0001_JPEG.jpg 改名为 0001.jpg
            new_name = fname.replace("_JPEG", "")
            dst_path = os.path.join(dst_jpeg_dir, new_name)
        else:
            # 复制到 test_original
            dst_path = os.path.join(dst_orig_dir, fname)

        shutil.copy2(src_path, dst_path)

print("Done.")

Splitting real: 100%|██████████| 12013/12013 [29:11<00:00,  6.86it/s]

Done.


In [ ]:
import os

for root_name in ["test_original", "test_jpeg"]:
    root = f"/content/drive/MyDrive/MLP/Project5/dataset_expand/{root_name}"
    print(root_name)

    for cls in ["fake", "real"]:
        cls_dir = os.path.join(root, cls)
        files = [
            f for f in os.listdir(cls_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
        ]
        print(cls, len(files), files[:3])

    print()

test_original
fake 6000 ['5510.jpg', '5492.jpg', '5499.jpg']
real 6008 ['5509.jpg', '5508.jpg', '5510.jpg']

test_jpeg
fake 6000 ['5522.jpg', '5489.jpg', '5508.jpg']
real 6005 ['5505.jpg', '5511.jpg', '5519.jpg']



In [ ]:
import os

orig_root = "/content/drive/MyDrive/MLP/Project5/dataset_expand/test_original"
jpeg_root = "/content/drive/MyDrive/MLP/Project5/dataset_expand/test_jpeg"

valid_exts = (".jpg", ".jpeg", ".png", ".webp")

def get_stems(folder):
    stems = set()
    files = []

    for fname in os.listdir(folder):
        if fname.lower().endswith(valid_exts):
            stem = os.path.splitext(fname)[0]
            stems.add(stem)
            files.append(fname)

    return stems, files

for cls in ["fake", "real"]:
    orig_dir = os.path.join(orig_root, cls)
    jpeg_dir = os.path.join(jpeg_root, cls)

    orig_stems, orig_files = get_stems(orig_dir)
    jpeg_stems, jpeg_files = get_stems(jpeg_dir)

    only_orig = sorted(list(orig_stems - jpeg_stems))
    only_jpeg = sorted(list(jpeg_stems - orig_stems))
    paired = sorted(list(orig_stems & jpeg_stems))

    print("=" * 50)
    print(cls)
    print("original:", len(orig_stems))
    print("jpeg:", len(jpeg_stems))
    print("paired:", len(paired))
    print("only in original:", len(only_orig))
    print(only_orig[:30])
    print("only in jpeg:", len(only_jpeg))
    print(only_jpeg[:30])

fake
original: 6000
jpeg: 6000
paired: 6000
only in original: 0
[]
only in jpeg: 0
[]
real
original: 6008
jpeg: 6005
paired: 6005
only in original: 3
['5197', '5325', '5879']
only in jpeg: 0
[]


In [ ]:
import os
import shutil

orig_root = "/content/drive/MyDrive/MLP/Project5/dataset_expand/test_original"
jpeg_root = "/content/drive/MyDrive/MLP/Project5/dataset_expand/test_jpeg"

exclude_root = "/content/drive/MyDrive/MLP/Project5/dataset_expand/excluded_unpaired"

valid_exts = (".jpg", ".jpeg", ".png", ".webp")

def get_stem_to_file(folder):
    mapping = {}

    for fname in os.listdir(folder):
        if fname.lower().endswith(valid_exts):
            stem = os.path.splitext(fname)[0]
            mapping[stem] = fname

    return mapping

for cls in ["fake", "real"]:
    orig_dir = os.path.join(orig_root, cls)
    jpeg_dir = os.path.join(jpeg_root, cls)

    orig_map = get_stem_to_file(orig_dir)
    jpeg_map = get_stem_to_file(jpeg_dir)

    orig_stems = set(orig_map.keys())
    jpeg_stems = set(jpeg_map.keys())

    only_orig = sorted(list(orig_stems - jpeg_stems))
    only_jpeg = sorted(list(jpeg_stems - orig_stems))

    # move original-only files
    dst_orig_exclude = os.path.join(exclude_root, "test_original", cls)
    os.makedirs(dst_orig_exclude, exist_ok=True)

    for stem in only_orig:
        fname = orig_map[stem]
        src = os.path.join(orig_dir, fname)
        dst = os.path.join(dst_orig_exclude, fname)
        shutil.move(src, dst)
        print("Moved original-only:", src, "->", dst)

    # move jpeg-only files
    dst_jpeg_exclude = os.path.join(exclude_root, "test_jpeg", cls)
    os.makedirs(dst_jpeg_exclude, exist_ok=True)

    for stem in only_jpeg:
        fname = jpeg_map[stem]
        src = os.path.join(jpeg_dir, fname)
        dst = os.path.join(dst_jpeg_exclude, fname)
        shutil.move(src, dst)
        print("Moved jpeg-only:", src, "->", dst)

print("Done moving unpaired files.")

Moved original-only: /content/drive/MyDrive/MLP/Project5/dataset_expand/test_original/real/5197.jpg -> /content/drive/MyDrive/MLP/Project5/dataset_expand/excluded_unpaired/test_original/real/5197.jpg
Moved original-only: /content/drive/MyDrive/MLP/Project5/dataset_expand/test_original/real/5325.jpg -> /content/drive/MyDrive/MLP/Project5/dataset_expand/excluded_unpaired/test_original/real/5325.jpg
Moved original-only: /content/drive/MyDrive/MLP/Project5/dataset_expand/test_original/real/5879.jpg -> /content/drive/MyDrive/MLP/Project5/dataset_expand/excluded_unpaired/test_original/real/5879.jpg
Done moving unpaired files.


In [ ]:
import os

for root_name in ["test_original", "test_jpeg"]:
    root = f"/content/drive/MyDrive/MLP/Project5/dataset_expand/{root_name}"
    print(root_name)

    for cls in ["fake", "real"]:
        cls_dir = os.path.join(root, cls)
        files = [
            f for f in os.listdir(cls_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
        ]
        print(cls, len(files))

    print()

test_original
fake 6000
real 6005

test_jpeg
fake 6000
real 6005



In [ ]:
import os
import shutil

output_dir = "/content/drive/MyDrive/MLP/Project5/final_outputs/resnet18_baseline_expand"
backup_dir = os.path.join(output_dir, "old_eval_before_test_split_fix")
os.makedirs(backup_dir, exist_ok=True)

old_files = [
    "metrics_original.json",
    "metrics_jpeg.json",
    "predictions_original.csv",
    "predictions_jpeg.csv",
    "summary.csv",
]

for fname in old_files:
    src = os.path.join(output_dir, fname)
    dst = os.path.join(backup_dir, fname)
    if os.path.exists(src):
        shutil.move(src, dst)
        print("Moved:", src, "->", dst)

print("Done.")

Moved: /content/drive/MyDrive/MLP/Project5/final_outputs/resnet18_baseline_expand/metrics_original.json -> /content/drive/MyDrive/MLP/Project5/final_outputs/resnet18_baseline_expand/old_eval_before_test_split_fix/metrics_original.json
Moved: /content/drive/MyDrive/MLP/Project5/final_outputs/resnet18_baseline_expand/metrics_jpeg.json -> /content/drive/MyDrive/MLP/Project5/final_outputs/resnet18_baseline_expand/old_eval_before_test_split_fix/metrics_jpeg.json
Moved: /content/drive/MyDrive/MLP/Project5/final_outputs/resnet18_baseline_expand/predictions_original.csv -> /content/drive/MyDrive/MLP/Project5/final_outputs/resnet18_baseline_expand/old_eval_before_test_split_fix/predictions_original.csv
Moved: /content/drive/MyDrive/MLP/Project5/final_outputs/resnet18_baseline_expand/predictions_jpeg.csv -> /content/drive/MyDrive/MLP/Project5/final_outputs/resnet18_baseline_expand/old_eval_before_test_split_fix/predictions_jpeg.csv
Done.


In [ ]:
import os

data_root = "/content/drive/MyDrive/MLP/Project5/dataset_expand"

for split in ["train", "val"]:
    print(split)
    for cls in ["fake", "real"]:
        cls_dir = os.path.join(data_root, split, cls)
        files = [
            f for f in os.listdir(cls_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
        ]
        jpeg_tagged = [f for f in files if "_JPEG" in f]
        print(cls, "total:", len(files), "JPEG-tagged:", len(jpeg_tagged))
    print()

train
fake total: 4998 JPEG-tagged: 0
real total: 4998 JPEG-tagged: 0

val
fake total: 1002 JPEG-tagged: 0
real total: 1002 JPEG-tagged: 0



验证expanded dataset 的 JPEG 退化情况

In [ ]:
#
import os
import random
import math
import pandas as pd
import numpy as np
from PIL import Image, ImageChops, ImageStat

orig_root = "/content/drive/MyDrive/MLP/Project5/dataset_expand/test_original"
jpeg_root = "/content/drive/MyDrive/MLP/Project5/dataset_expand/test_jpeg"

valid_exts = (".jpg", ".jpeg", ".png", ".webp")

def calc_psnr(img1, img2):
    img1 = img1.convert("RGB")
    img2 = img2.convert("RGB")
    if img1.size != img2.size:
        img2 = img2.resize(img1.size)

    diff = ImageChops.difference(img1, img2)
    stat = ImageStat.Stat(diff)
    mse = sum(v ** 2 for v in stat.rms) / 3

    if mse == 0:
        return float("inf"), 0.0

    psnr = 10 * math.log10((255 ** 2) / mse)
    mean_abs_diff = sum(stat.mean) / 3

    return psnr, mean_abs_diff


pairs = []

for cls in ["fake", "real"]:
    orig_dir = os.path.join(orig_root, cls)
    jpeg_dir = os.path.join(jpeg_root, cls)

    orig_files = [
        f for f in os.listdir(orig_dir)
        if f.lower().endswith(valid_exts)
    ]

    for fname in orig_files:
        orig_path = os.path.join(orig_dir, fname)
        jpeg_path = os.path.join(jpeg_dir, fname)

        if os.path.exists(jpeg_path):
            pairs.append((cls, fname, orig_path, jpeg_path))

print("Total pairs:", len(pairs))

sample_pairs = random.sample(pairs, min(500, len(pairs)))

rows = []

for cls, fname, orig_path, jpeg_path in sample_pairs:
    try:
        img_orig = Image.open(orig_path)
        img_jpeg = Image.open(jpeg_path)

        psnr, mean_abs_diff = calc_psnr(img_orig, img_jpeg)

        orig_size = os.path.getsize(orig_path)
        jpeg_size = os.path.getsize(jpeg_path)

        rows.append({
            "class": cls,
            "filename": fname,
            "orig_size_kb": orig_size / 1024,
            "jpeg_size_kb": jpeg_size / 1024,
            "size_ratio": jpeg_size / orig_size,
            "psnr": psnr,
            "mean_abs_diff": mean_abs_diff,
            "orig_path": orig_path,
            "jpeg_path": jpeg_path,
        })

    except Exception as e:
        print("Error:", fname, e)

df = pd.DataFrame(rows)

print(df[["orig_size_kb", "jpeg_size_kb", "size_ratio", "psnr", "mean_abs_diff"]].describe())

print("\nSize ratio > 1:", (df["size_ratio"] > 1).sum(), "/", len(df))
print("PSNR > 40:", (df["psnr"] > 40).sum(), "/", len(df))
print("PSNR > 45:", (df["psnr"] > 45).sum(), "/", len(df))
print("PSNR < 35:", (df["psnr"] < 35).sum(), "/", len(df))

df.to_csv(
    "/content/drive/MyDrive/MLP/Project5/final_outputs/jpeg_quality_check_sample.csv",
    index=False,
    encoding="utf-8-sig"
)

df.sort_values("psnr", ascending=False).head(10)

Total pairs: 9884
       orig_size_kb  jpeg_size_kb  size_ratio        psnr  mean_abs_diff
count    500.000000    500.000000  500.000000  500.000000     500.000000
mean     842.380176    968.046082    1.332869   47.610200       0.685393
std     1684.199376   1851.763494    0.241656    4.270445       0.518120
min       15.159180     19.970703    0.246369   32.917702       0.041282
25%      103.144287    144.523193    1.189315   45.131588       0.361595
50%      174.232422    243.327148    1.403567   47.907592       0.541444
75%      500.601807    652.755615    1.439340   50.522558       0.842477
max    13687.927734  15059.373047    2.517200   59.889171       3.741278

Size ratio > 1: 476 / 500
PSNR > 40: 478 / 500
PSNR > 45: 380 / 500
PSNR < 35: 3 / 500


,class,filename,orig_size_kb,jpeg_size_kb,size_ratio,psnr,mean_abs_diff,orig_path,jpeg_path
429,real,2709.jpg,20.326172,24.873047,1.223696,59.889171,0.062011,/content/drive/MyDrive/MLP/Project5/dataset_ex...,/content/drive/MyDrive/MLP/Project5/dataset_ex...
238,real,3649.jpg,36.609375,41.296875,1.128041,59.078176,0.041282,/content/drive/MyDrive/MLP/Project5/dataset_ex...,/content/drive/MyDrive/MLP/Project5/dataset_ex...
199,real,4641.jpg,1012.395508,1380.579102,1.363676,58.028708,0.082650,/content/drive/MyDrive/MLP/Project5/dataset_ex...,/content/drive/MyDrive/MLP/Project5/dataset_ex...
53,real,2358.jpg,22.559570,27.959961,1.239384,57.739850,0.066440,/content/drive/MyDrive/MLP/Project5/dataset_ex...,/content/drive/MyDrive/MLP/Project5/dataset_ex...
242,real,5965.jpg,351.769531,429.335938,1.220503,57.154839,0.089557,/content/drive/MyDrive/MLP/Project5/dataset_ex...,/content/drive/MyDrive/MLP/Project5/dataset_ex...
316,real,2171.jpg,2079.587891,2514.958008,1.209354,56.963104,0.111349,/content/drive/MyDrive/MLP/Project5/dataset_ex...,/content/drive/MyDrive/MLP/Project5/dataset_ex...
118,real,2593.jpg,62.799805,81.494141,1.297681,56.114992,0.096381,/content/drive/MyDrive/MLP/Project5/dataset_ex...,/content/drive/MyDrive/MLP/Project5/dataset_ex...
482,real,5455.jpg,1249.583008,1559.913086,1.248347,56.008640,0.122159,/content/drive/MyDrive/MLP/Project5/dataset_ex...,/content/drive/MyDrive/MLP/Project5/dataset_ex...
23,real,5331.jpg,1215.751953,1424.770508,1.171925,55.970725,0.128696,/content/drive/MyDrive/MLP/Project5/dataset_ex...,/content/drive/MyDrive/MLP/Project5/dataset_ex...
321,real,3746.jpg,50.526367,63.678711,1.260307,55.955806,0.096830,/content/drive/MyDrive/MLP/Project5/dataset_ex...,/content/drive/MyDrive/MLP/Project5/dataset_ex...


In [ ]:
# ============================================================
# Quick check: old dataset JPEG degradation strength
# ============================================================

import os, random, math
import pandas as pd
from PIL import Image, ImageChops, ImageStat

old_orig_root = "/content/drive/MyDrive/MLP/Project5/期中资料/dataset_project5/test"
old_jpeg_root = "/content/drive/MyDrive/MLP/Project5/期中资料/dataset_project5/test_jpeg_q30"

valid_exts = (".jpg", ".jpeg", ".png", ".webp")

def norm_id(fname):
    stem = os.path.splitext(fname)[0]
    return stem.replace("_JPEG", "").replace("_jpeg", "").replace("_q30", "").replace("_Q30", "")

def psnr_and_diff(img1, img2):
    img1 = img1.convert("RGB")
    img2 = img2.convert("RGB")
    if img1.size != img2.size:
        img2 = img2.resize(img1.size)

    diff = ImageChops.difference(img1, img2)
    stat = ImageStat.Stat(diff)
    mse = sum(v ** 2 for v in stat.rms) / 3
    mean_abs_diff = sum(stat.mean) / 3

    if mse == 0:
        return float("inf"), mean_abs_diff
    psnr = 10 * math.log10((255 ** 2) / mse)
    return psnr, mean_abs_diff

pairs = []

for cls in ["fake", "real"]:
    orig_dir = os.path.join(old_orig_root, cls)
    jpeg_dir = os.path.join(old_jpeg_root, cls)

    orig_files = [f for f in os.listdir(orig_dir) if f.lower().endswith(valid_exts)]
    jpeg_files = [f for f in os.listdir(jpeg_dir) if f.lower().endswith(valid_exts)]

    orig_map = {norm_id(f): f for f in orig_files}
    jpeg_map = {norm_id(f): f for f in jpeg_files}

    paired_ids = sorted(set(orig_map.keys()) & set(jpeg_map.keys()))

    print(f"{cls}: original={len(orig_files)}, jpeg={len(jpeg_files)}, paired={len(paired_ids)}")

    for image_id in paired_ids:
        pairs.append((
            cls,
            image_id,
            os.path.join(orig_dir, orig_map[image_id]),
            os.path.join(jpeg_dir, jpeg_map[image_id])
        ))

print("Total pairs:", len(pairs))

sample_pairs = random.sample(pairs, min(500, len(pairs)))

rows = []

for cls, image_id, orig_path, jpeg_path in sample_pairs:
    try:
        img_orig = Image.open(orig_path)
        img_jpeg = Image.open(jpeg_path)

        psnr, mean_abs_diff = psnr_and_diff(img_orig, img_jpeg)

        orig_size = os.path.getsize(orig_path)
        jpeg_size = os.path.getsize(jpeg_path)

        rows.append({
            "class": cls,
            "image_id": image_id,
            "orig_size_kb": orig_size / 1024,
            "jpeg_size_kb": jpeg_size / 1024,
            "size_ratio": jpeg_size / orig_size,
            "psnr": psnr,
            "mean_abs_diff": mean_abs_diff,
            "orig_path": orig_path,
            "jpeg_path": jpeg_path,
        })

    except Exception as e:
        print("Error:", image_id, e)

old_df = pd.DataFrame(rows)

print("\n=== Old dataset JPEG quality summary ===")
display(old_df[["orig_size_kb", "jpeg_size_kb", "size_ratio", "psnr", "mean_abs_diff"]].describe())

print("\nSize ratio > 1:", (old_df["size_ratio"] > 1).sum(), "/", len(old_df))
print("PSNR > 40:", (old_df["psnr"] > 40).sum(), "/", len(old_df))
print("PSNR > 45:", (old_df["psnr"] > 45).sum(), "/", len(old_df))
print("PSNR < 35:", (old_df["psnr"] < 35).sum(), "/", len(old_df))

save_path = "/content/drive/MyDrive/MLP/Project5/final_outputs/old_dataset_jpeg_quality_check_sample.csv"
old_df.to_csv(save_path, index=False, encoding="utf-8-sig")
print("\nSaved to:", save_path)

print("\nTop 5 weakest degradation, highest PSNR:")
display(old_df.sort_values("psnr", ascending=False).head(5))

print("\nTop 5 strongest degradation, lowest PSNR:")
display(old_df.sort_values("psnr", ascending=True).head(5))

fake: original=450, jpeg=450, paired=450
real: original=450, jpeg=450, paired=450
Total pairs: 900

=== Old dataset JPEG quality summary ===


,orig_size_kb,jpeg_size_kb,size_ratio,psnr,mean_abs_diff
count,500.000000,500.000000,500.000000,500.000000,500.000000
mean,822.557410,231.790637,0.367101,32.465185,4.417064
std,1368.391368,398.975086,0.158917,3.658136,2.014131
min,20.845703,9.260742,0.026749,23.698973,1.215560
25%,135.572510,53.285400,0.242710,29.987264,2.963596
50%,242.084473,80.103027,0.420681,32.325750,3.987478
75%,748.255615,187.225586,0.496047,34.939380,5.390529
max,11164.948242,3436.083984,0.749268,42.354220,12.780993



Size ratio > 1: 0 / 500
PSNR > 40: 9 / 500
PSNR > 45: 0 / 500
PSNR < 35: 377 / 500

Saved to: /content/drive/MyDrive/MLP/Project5/final_outputs/old_dataset_jpeg_quality_check_sample.csv

Top 5 weakest degradation, highest PSNR:


,class,image_id,orig_size_kb,jpeg_size_kb,size_ratio,psnr,mean_abs_diff,orig_path,jpeg_path
52,real,2171,2079.587891,787.682617,0.378769,42.354220,1.323146,/content/drive/MyDrive/MLP/Project5/期中资料/datas...,/content/drive/MyDrive/MLP/Project5/期中资料/datas...
333,real,4582,652.409180,478.262695,0.733072,42.334015,1.447105,/content/drive/MyDrive/MLP/Project5/期中资料/datas...,/content/drive/MyDrive/MLP/Project5/期中资料/datas...
395,real,5313,910.229492,369.934570,0.406419,41.637293,1.215560,/content/drive/MyDrive/MLP/Project5/期中资料/datas...,/content/drive/MyDrive/MLP/Project5/期中资料/datas...
271,real,3157,37.200195,19.582031,0.526396,41.609547,1.589560,/content/drive/MyDrive/MLP/Project5/期中资料/datas...,/content/drive/MyDrive/MLP/Project5/期中资料/datas...
474,real,1623,226.408203,111.490234,0.492430,41.540667,1.597189,/content/drive/MyDrive/MLP/Project5/期中资料/datas...,/content/drive/MyDrive/MLP/Project5/期中资料/datas...



Top 5 strongest degradation, lowest PSNR:


,class,image_id,orig_size_kb,jpeg_size_kb,size_ratio,psnr,mean_abs_diff,orig_path,jpeg_path
449,real,0054,344.297852,124.197266,0.360726,23.698973,12.402775,/content/drive/MyDrive/MLP/Project5/期中资料/datas...,/content/drive/MyDrive/MLP/Project5/期中资料/datas...
2,fake,2327,990.970703,186.299805,0.187997,23.706750,12.780993,/content/drive/MyDrive/MLP/Project5/期中资料/datas...,/content/drive/MyDrive/MLP/Project5/期中资料/datas...
274,real,3943,415.558594,156.462891,0.376512,23.833759,11.790597,/content/drive/MyDrive/MLP/Project5/期中资料/datas...,/content/drive/MyDrive/MLP/Project5/期中资料/datas...
40,real,4876,208.478516,83.709961,0.401528,24.118417,11.611264,/content/drive/MyDrive/MLP/Project5/期中资料/datas...,/content/drive/MyDrive/MLP/Project5/期中资料/datas...
90,real,3024,465.759766,210.645508,0.452262,24.413957,10.128345,/content/drive/MyDrive/MLP/Project5/期中资料/datas...,/content/drive/MyDrive/MLP/Project5/期中资料/datas...


In [ ]:
from PIL import Image
import os
from tqdm import tqdm

src_root = "/content/drive/MyDrive/MLP/Project5/dataset_expand/test_original"
dst_root = "/content/drive/MyDrive/MLP/Project5/dataset_expand/test_jpeg_q30_controlled"

quality = 30
classes = ["fake", "real"]
valid_exts = (".jpg", ".jpeg", ".png", ".webp")

for cls in classes:
    src_dir = os.path.join(src_root, cls)
    dst_dir = os.path.join(dst_root, cls)
    os.makedirs(dst_dir, exist_ok=True)

    files = [
        f for f in os.listdir(src_dir)
        if f.lower().endswith(valid_exts)
    ]

    for fname in tqdm(files, desc=f"{cls} q{quality}"):
        src_path = os.path.join(src_dir, fname)
        dst_path = os.path.join(dst_dir, os.path.splitext(fname)[0] + ".jpg")

        img = Image.open(src_path).convert("RGB")
        img.save(
            dst_path,
            "JPEG",
            quality=quality,
            optimize=True,
            subsampling=2
        )

print("Done:", dst_root)

fake q30:  50%|████▉     | 2999/6000 [05:00<00:50, 59.77it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
real q30:  79%|███████▉  | 4730/6005 [15:16<10:12,  2.08it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (143040000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
real q30:  79%|███████▉  | 4735/6005 [15:18<06:27,  3.27it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (121554000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
real q30: 100%|██████████| 6005/6005 [17:35<00:00,  5.69it/s]

Done: /content/drive/MyDrive/MLP/Project5/dataset_expand/test_jpeg_q30_controlled


In [ ]:
import os
import shutil

output_dir = "/content/drive/MyDrive/MLP/Project5/final_outputs/resnet18_baseline_expand"

archive_dir = os.path.join(output_dir, "eval_jpeg_reencoded")
os.makedirs(archive_dir, exist_ok=True)

rename_map = {
    "metrics_original.json": "metrics_original.json",
    "metrics_jpeg.json": "metrics_jpeg_reencoded.json",
    "predictions_original.csv": "predictions_original.csv",
    "predictions_jpeg.csv": "predictions_jpeg_reencoded.csv",
    "summary.csv": "summary_jpeg_reencoded.csv",
}

for src_name, dst_name in rename_map.items():
    src = os.path.join(output_dir, src_name)
    dst = os.path.join(archive_dir, dst_name)

    if os.path.exists(src):
        shutil.move(src, dst)
        print("Moved:", src, "->", dst)
    else:
        print("Not found:", src)

print("Done archiving current re-encoded JPEG evaluation.")

Moved: /content/drive/MyDrive/MLP/Project5/final_outputs/resnet18_baseline_expand/metrics_original.json -> /content/drive/MyDrive/MLP/Project5/final_outputs/resnet18_baseline_expand/eval_jpeg_reencoded/metrics_original.json
Moved: /content/drive/MyDrive/MLP/Project5/final_outputs/resnet18_baseline_expand/metrics_jpeg.json -> /content/drive/MyDrive/MLP/Project5/final_outputs/resnet18_baseline_expand/eval_jpeg_reencoded/metrics_jpeg_reencoded.json
Moved: /content/drive/MyDrive/MLP/Project5/final_outputs/resnet18_baseline_expand/predictions_original.csv -> /content/drive/MyDrive/MLP/Project5/final_outputs/resnet18_baseline_expand/eval_jpeg_reencoded/predictions_original.csv
Moved: /content/drive/MyDrive/MLP/Project5/final_outputs/resnet18_baseline_expand/predictions_jpeg.csv -> /content/drive/MyDrive/MLP/Project5/final_outputs/resnet18_baseline_expand/eval_jpeg_reencoded/predictions_jpeg_reencoded.csv
Moved: /content/drive/MyDrive/MLP/Project5/final_outputs/resnet18_baseline_expand/summar

[草稿2]

In [36]:
# ============================================================
# Check corrupted / truncated images before training
# ============================================================

import os
import pandas as pd
from PIL import Image, ImageFile
from tqdm import tqdm

# 先不要允许 truncated image，目的是把潜在问题图找出来
ImageFile.LOAD_TRUNCATED_IMAGES = False

data_root = CONFIG["data_root"]

splits = [
    CONFIG["train_dir"],
    CONFIG["val_dir"],
    CONFIG["test_orig_dir"],
    CONFIG["test_jpeg_dir"],
]

valid_exts = (".jpg", ".jpeg", ".png", ".webp")

bad_rows = []
all_count = 0

for split in splits:
    split_root = os.path.join(data_root, split)

    for cls in ["fake", "real"]:
        cls_dir = os.path.join(split_root, cls)

        if not os.path.exists(cls_dir):
            print("Missing folder:", cls_dir)
            continue

        files = [
            f for f in os.listdir(cls_dir)
            if f.lower().endswith(valid_exts)
        ]

        for fname in tqdm(files, desc=f"Checking {split}/{cls}"):
            path = os.path.join(cls_dir, fname)
            all_count += 1

            try:
                with Image.open(path) as img:
                    img.verify()

                # verify() 后需要重新打开一次，测试能不能真正 load + convert RGB
                with Image.open(path) as img:
                    img.load()
                    img.convert("RGB")

            except Exception as e:
                bad_rows.append({
                    "split": split,
                    "class": cls,
                    "filename": fname,
                    "path": path,
                    "error_type": type(e).__name__,
                    "error_message": str(e),
                })

bad_df = pd.DataFrame(bad_rows)

print("=" * 60)
print("Total checked:", all_count)
print("Bad images:", len(bad_df))

if len(bad_df) > 0:
    display(bad_df)

    save_path = os.path.join(
        CONFIG["output_dir"],
        "bad_images_report.csv"
    )
    bad_df.to_csv(save_path, index=False, encoding="utf-8-sig")
    print("Saved bad image report to:", save_path)
else:
    print("No bad images found.")

Checking train/fake:   1%|          | 49/4998 [01:02<1:44:37,  1.27s/it]


KeyboardInterrupt: 